# Ropedia Academy — A4 · 3D human mesh recovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A4.ipynb)

> **The HMR recipe end-to-end: regress (β, θ, camera), supervise 3D through a differentiable reprojection loss, and add a pose prior to break depth ambiguity.**
>
> 完整的 HMR 配方：回归 (β、θ、相机)，经可微重投影损失用 2D 监督 3D，并加姿态先验打破深度歧义。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A4

In [ ]:
import torch, torch.nn as nn

# HMR: regress SMPL params + camera from an image feature, train with NO 3D labels.
img_feat = torch.randn(1, 512)                  # from a CNN/ViT backbone
head = nn.Linear(512, 10 + 24*3 + 3)            # beta(10) + theta(72) + camera(s,tx,ty)
out = head(img_feat)[0]
beta, theta, cam = out[:10], out[10:82], out[82:]
joints3d = theta.reshape(24, 3)                 # stand-in for SMPL(beta,theta) joints

def project(j3d, cam):                           # weak-perspective projection
    s, t = cam[0], cam[1:]
    return s * j3d[:, :2] + t                    # 3D joints -> 2D pixels

kp2d_gt = torch.randn(24, 2)                     # abundant 2D keypoint labels
reproj  = ((project(joints3d, cam) - kp2d_gt) ** 2).mean()   # supervise 3D via 2D
prior   = (theta ** 2).mean()                    # stand-in for an adversarial pose prior
loss    = reproj + 0.1 * prior                   # reprojection is under-constrained alone
loss.backward()
print("reprojection loss:", round(reproj.item(), 3))
print("trainable from 2D keypoints only:", head.weight.grad is not None)

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A4
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks